In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# FIR LATTICE IMPLEMENTATION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'
MAX_STAGES = 4
N = 80

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.lat-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.lat-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.lat-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.lat-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.lat-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.lat-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.lat-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="lat-root">

<div class="lat-header">
FIR Lattice Implementation — Building the Filter Stage by Stage
</div>

<div class="lat-doc">

An FIR lattice filter is constructed by connecting identical two-input,
two-output stages in cascade. The number of stages is equal to the order
of the FIR filter.

For stage m,

<div class="lat-equation">
<b>
f<sub>m</sub>[n]
=
f<sub>m−1</sub>[n]
+
K<sub>m</sub> g<sub>m−1</sub>[n−1]
</b>
</div>

<div class="lat-equation">
<b>
g<sub>m</sub>[n]
=
K<sub>m</sub> f<sub>m−1</sub>[n]
+
g<sub>m−1</sub>[n−1].
</b>
</div>

The initial signals are

<div class="lat-equation">
<b>
f₀[n] = g₀[n] = x[n].
</b>
</div>

The coefficients K₁, K₂, ... are the <b>reflection coefficients</b>.
After m stages, the upper output f<sub>m</sub>[n] is identical to the
output of an FIR filter of order m.

Only the reflection coefficients associated with the currently active
stages are enabled.

The equivalent direct-form FIR coefficients are not predefined in this
notebook. They are calculated recursively from the reflection coefficients.

<div class="lat-equation">
<b>
different internal structure → same FIR system → same output
</b>
</div>

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

stage_slider = IntSlider(value=4,min=1,max=MAX_STAGES,step=1,description='Stages:',continuous_update=True,style={'description_width':'45px'},layout=Layout(width='170px'))

K1_slider = FloatSlider(value=0.35,min=-0.85,max=0.85,step=0.05,description='K₁:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K2_slider = FloatSlider(value=-0.30,min=-0.85,max=0.85,step=0.05,description='K₂:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K3_slider = FloatSlider(value=0.25,min=-0.85,max=0.85,step=0.05,description='K₃:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K4_slider = FloatSlider(value=-0.20,min=-0.85,max=0.85,step=0.05,description='K₄:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K_sliders = [K1_slider,K2_slider,K3_slider,K4_slider]

controls = HBox([stage_slider,K1_slider,K2_slider,K3_slider,K4_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 8px',margin='0 0 3px 0'))

# ============================================================
# TEST INPUT
# ============================================================

n = np.arange(N)

x = np.zeros(N)

x[0] = 1.0

x += 0.30*np.sin(0.18*np.pi*n)

x += 0.15*np.sin(0.55*np.pi*n)

# ============================================================
# REFLECTION COEFFICIENTS
# ============================================================

def get_reflection_coefficients():

    return np.array([K1_slider.value,K2_slider.value,K3_slider.value,K4_slider.value])

# ============================================================
# FIR COEFFICIENTS FROM REFLECTION COEFFICIENTS
# ============================================================

def reflection_to_fir(K):

    A = np.array([1.0])
    B = np.array([1.0])

    A_history = [A.copy()]
    B_history = [B.copy()]

    for Km in K:

        A_pad = np.append(A,0.0)
        B_shift = np.insert(B,0,0.0)

        A_new = A_pad+Km*B_shift
        B_new = Km*A_pad+B_shift

        A = A_new
        B = B_new

        A_history.append(A.copy())
        B_history.append(B.copy())

    return A,B,A_history,B_history

# ============================================================
# FIR LATTICE CALCULATION
# ============================================================

def lattice_filter(x,K):

    f = x.copy()
    g = x.copy()

    f_history = [f.copy()]
    g_history = [g.copy()]

    for Km in K:

        g_delay = np.zeros_like(g)
        g_delay[1:] = g[:-1]

        f_new = f+Km*g_delay
        g_new = Km*f+g_delay

        f = f_new
        g = g_new

        f_history.append(f.copy())
        g_history.append(g.copy())

    return f,g,f_history,g_history

# ============================================================
# PRECOMPUTE FIXED AXIS LIMITS
# ============================================================

test_values = [-0.85,0.85]

all_internal_values = []
all_output_values = []

for K1 in test_values:

    for K2 in test_values:

        for K3 in test_values:

            for K4 in test_values:

                K_test = np.array([K1,K2,K3,K4])

                A_test,B_test,A_hist_test,B_hist_test = reflection_to_fir(K_test)

                y_lattice_test,g_test,f_hist_test,g_hist_test = lattice_filter(x,K_test)

                y_direct_test = signal.lfilter(A_test,[1.0],x)

                for f_test in f_hist_test:

                    all_internal_values.extend(f_test)

                for g_test_stage in g_hist_test:

                    all_internal_values.extend(g_test_stage)

                all_output_values.extend(y_direct_test)

                all_output_values.extend(y_lattice_test)

internal_limit = 1.10*max(np.max(np.abs(all_internal_values)),1.0)

output_limit = 1.10*max(np.max(np.abs(all_output_values)),1.0)

# ============================================================
# INITIAL VALUES
# ============================================================

active_stages = stage_slider.value

K_all = get_reflection_coefficients()

K = K_all[:active_stages]

A,B,A_history,B_history = reflection_to_fir(K)

y_lattice,g_final,f_history,g_history = lattice_filter(x,K)

y_direct = signal.lfilter(A,[1.0],x)

difference = y_direct-y_lattice

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# FIGURE 1 — LATTICE STRUCTURE
# ============================================================

fig1,ax_structure = plt.subplots(figsize=(9.0,3.0))

fig1.canvas.toolbar_visible = False
fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False

ax_structure.set_xlim(0,11)
ax_structure.set_ylim(-2.0,2.0)
ax_structure.axis('off')
ax_structure.set_title('FIR Lattice Structure')

stage_centers = [2.4,4.5,6.6,8.7]

stage_rectangles = []
stage_labels = []
upper_labels = []
lower_labels = []

ax_structure.text(0.25,0.95,r'$f_0[n]=x[n]$',fontsize=10.5,fontweight='bold')
ax_structure.text(0.25,-0.95,r'$g_0[n]=x[n]$',fontsize=10.5,fontweight='bold')

ax_structure.annotate('',xy=(1.40,0.75),xytext=(0.95,0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})
ax_structure.annotate('',xy=(1.40,-0.75),xytext=(0.95,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

for m,xc in enumerate(stage_centers):

    rect = plt.Rectangle((xc-0.72,-1.35),1.44,2.70,fill=False,linewidth=1.3)

    ax_structure.add_patch(rect)

    stage_rectangles.append(rect)

    label = ax_structure.text(xc,1.55,f'Stage {m+1}',ha='center',fontsize=10.5,fontweight='bold')

    stage_labels.append(label)

    upper = ax_structure.text(xc,0.32,'',ha='center',va='center',fontsize=10)

    lower = ax_structure.text(xc,-0.32,'',ha='center',va='center',fontsize=10)

    upper_labels.append(upper)
    lower_labels.append(lower)

    ax_structure.plot([xc-0.60,xc+0.60],[0.75,0.75],linewidth=1.2)
    ax_structure.plot([xc-0.60,xc+0.60],[-0.75,-0.75],linewidth=1.2)

    ax_structure.annotate('',xy=(xc+0.55,0.75),xytext=(xc-0.55,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.0})
    ax_structure.annotate('',xy=(xc+0.55,-0.75),xytext=(xc-0.55,0.75),arrowprops={'arrowstyle':'->','linewidth':1.0})

    ax_structure.text(xc-0.30,-1.10,r'$z^{-1}$',ha='center',fontsize=9.5)

    if m < MAX_STAGES-1:

        next_x = stage_centers[m+1]

        ax_structure.annotate('',xy=(next_x-0.72,0.75),xytext=(xc+0.72,0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})
        ax_structure.annotate('',xy=(next_x-0.72,-0.75),xytext=(xc+0.72,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})

ax_structure.annotate('',xy=(10.55,0.75),xytext=(9.42,0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.text(10.72,0.93,r'$y[n]=f_m[n]$',ha='center',fontsize=10.5,fontweight='bold')

plt.subplots_adjust(left=0.02,right=0.98,top=0.85,bottom=0.05)

# ============================================================
# FIGURE 2 — FIR COEFFICIENTS AND FREQUENCY RESPONSE
# ============================================================

fig2,(ax_coefficients,ax_response) = plt.subplots(1,2,figsize=(9.0,3.6))

fig2.canvas.toolbar_visible = False
fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False

coefficient_indices = np.arange(MAX_STAGES+1)

initial_coefficients = np.zeros(MAX_STAGES+1)

initial_coefficients[:len(A)] = A

coefficient_stems = ax_coefficients.vlines(coefficient_indices,0,initial_coefficients,linewidth=1.4)

coefficient_markers, = ax_coefficients.plot(coefficient_indices,initial_coefficients,'o',markersize=5)

ax_coefficients.axhline(0,linewidth=0.8)

ax_coefficients.set_xlim(-0.5,MAX_STAGES+0.5)
ax_coefficients.set_ylim(-2.5,2.5)
ax_coefficients.set_xticks(coefficient_indices)

ax_coefficients.set_title('Equivalent Direct-Form FIR Coefficients')
ax_coefficients.set_xlabel('Coefficient index k')
ax_coefficients.set_ylabel(r'$h_m[k]$')
ax_coefficients.grid(True,linestyle=':',alpha=0.25)

omega,H_direct = signal.freqz(A,[1.0],worN=1024)

response_line, = ax_response.plot(omega/np.pi,np.abs(H_direct),linewidth=1.4)

ax_response.set_xlim(0,1)
ax_response.set_ylim(0,5)

ax_response.set_title('Equivalent FIR Frequency Response')
ax_response.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax_response.set_ylabel(r'$|H(e^{j\omega})|$')
ax_response.grid(True,linestyle=':',alpha=0.25)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 3 — INTERNAL LATTICE SIGNALS
# ============================================================

fig3,(ax_forward,ax_backward) = plt.subplots(1,2,figsize=(9.0,3.6))

fig3.canvas.toolbar_visible = False
fig3.canvas.header_visible = False
fig3.canvas.footer_visible = False

forward_lines = []
backward_lines = []

for m in range(1,MAX_STAGES+1):

    f_initial = f_history[m] if m < len(f_history) else np.full(N,np.nan)
    g_initial = g_history[m] if m < len(g_history) else np.full(N,np.nan)

    line_f, = ax_forward.plot(n,f_initial,linewidth=1.1,label=rf'$f_{m}[n]$')
    line_g, = ax_backward.plot(n,g_initial,linewidth=1.1,label=rf'$g_{m}[n]$')

    forward_lines.append(line_f)
    backward_lines.append(line_g)

ax_forward.set_xlim(0,N-1)
ax_forward.set_ylim(-internal_limit,internal_limit)

ax_forward.set_title('Forward Lattice Signals')
ax_forward.set_xlabel('Sample index n')
ax_forward.set_ylabel(r'$f_m[n]$')
ax_forward.grid(True,linestyle=':',alpha=0.30)
ax_forward.legend(loc='upper right',ncol=2)

ax_backward.set_xlim(0,N-1)
ax_backward.set_ylim(-internal_limit,internal_limit)

ax_backward.set_title('Backward Lattice Signals')
ax_backward.set_xlabel('Sample index n')
ax_backward.set_ylabel(r'$g_m[n]$')
ax_backward.grid(True,linestyle=':',alpha=0.30)
ax_backward.legend(loc='upper right',ncol=2)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 4 — DIRECT VS LATTICE OUTPUT
# ============================================================

fig4,(ax_output,ax_difference) = plt.subplots(1,2,figsize=(9.0,3.5))

fig4.canvas.toolbar_visible = False
fig4.canvas.header_visible = False
fig4.canvas.footer_visible = False

direct_line, = ax_output.plot(n,y_direct,linewidth=1.5,label='Direct FIR implementation')

lattice_line, = ax_output.plot(n,y_lattice,'--',linewidth=1.3,label='Lattice implementation')

ax_output.set_xlim(0,N-1)
ax_output.set_ylim(-output_limit,output_limit)

ax_output.set_title('Final Output Comparison')
ax_output.set_xlabel('Sample index n')
ax_output.set_ylabel('y[n]')
ax_output.grid(True,linestyle=':',alpha=0.30)
ax_output.legend(loc='upper right')

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)
ax_difference.set_ylim(-1e-12,1e-12)

ax_difference.set_title('Numerical Difference')
ax_difference.set_xlabel('Sample index n')
ax_difference.set_ylabel(r'$y_D[n]-y_L[n]$')
ax_difference.grid(True,linestyle=':',alpha=0.30)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# UPDATE CALLBACK
# ============================================================

def update(change=None):

    active_stages = stage_slider.value

    # --------------------------------------------------------
    # ENABLE ONLY THE REFLECTION COEFFICIENTS OF ACTIVE STAGES
    # --------------------------------------------------------

    for i,slider in enumerate(K_sliders):

        slider.disabled = i >= active_stages

    # --------------------------------------------------------
    # CURRENT REFLECTION COEFFICIENTS
    # --------------------------------------------------------

    K_all = get_reflection_coefficients()

    K = K_all[:active_stages]

    A,B,A_history,B_history = reflection_to_fir(K)

    y_lattice,g_final,f_history,g_history = lattice_filter(x,K)

    y_direct = signal.lfilter(A,[1.0],x)

    difference = y_direct-y_lattice

    # --------------------------------------------------------
    # UPDATE LATTICE STRUCTURE
    # --------------------------------------------------------

    for m in range(MAX_STAGES):

        Km = K_all[m]

        upper_labels[m].set_text(rf'$K_{m+1}={Km:.2f}$')
        lower_labels[m].set_text(rf'$K_{m+1}={Km:.2f}$')

        if m < active_stages:

            stage_rectangles[m].set_alpha(1.0)
            stage_labels[m].set_alpha(1.0)
            upper_labels[m].set_alpha(1.0)
            lower_labels[m].set_alpha(1.0)

        else:

            stage_rectangles[m].set_alpha(0.18)
            stage_labels[m].set_alpha(0.18)
            upper_labels[m].set_alpha(0.18)
            lower_labels[m].set_alpha(0.18)

    # --------------------------------------------------------
    # UPDATE FIR COEFFICIENTS
    # --------------------------------------------------------

    coefficients = np.zeros(MAX_STAGES+1)

    coefficients[:len(A)] = A

    coefficient_stems.set_segments([[(k,0),(k,coefficients[k])] for k in range(MAX_STAGES+1)])

    coefficient_markers.set_ydata(coefficients)

    # --------------------------------------------------------
    # UPDATE FREQUENCY RESPONSE
    # --------------------------------------------------------

    omega,H = signal.freqz(A,[1.0],worN=1024)

    response_line.set_ydata(np.abs(H))

    # --------------------------------------------------------
    # UPDATE INTERNAL SIGNALS
    # --------------------------------------------------------

    for m in range(MAX_STAGES):

        if m+1 < len(f_history):

            forward_lines[m].set_ydata(f_history[m+1])
            backward_lines[m].set_ydata(g_history[m+1])

        else:

            forward_lines[m].set_ydata(np.full(N,np.nan))
            backward_lines[m].set_ydata(np.full(N,np.nan))

    # --------------------------------------------------------
    # UPDATE FINAL OUTPUTS
    # --------------------------------------------------------

    direct_line.set_ydata(y_direct)

    lattice_line.set_ydata(y_lattice)

    difference_line.set_ydata(difference)

    # --------------------------------------------------------
    # INFORMATION
    # --------------------------------------------------------

    maximum_difference = np.max(np.abs(difference))

    K_text = ', '.join([f'K{i+1} = {K[i]:.2f}' for i in range(active_stages)])

    h_text = ', '.join([f'{value:.6f}' for value in A])

    stage_rows = ""

    for m in range(1,active_stages+1):

        coeff_text = ', '.join([f'{v:.5f}' for v in A_history[m]])

        stage_rows += f"""
        <tr>
        <td style="padding:2px 10px;"><b>m = {m}</b></td>
        <td style="padding:2px 10px;">K<sub>{m}</sub> = {K[m-1]:.3f}</td>
        <td style="padding:2px 10px;">h<sub>{m}</sub> = [{coeff_text}]</td>
        </tr>
        """

    result_html.value = f"""
    <div class="lat-root">

    <div class="lat-box lat-result">

    <div class="lat-title">
    Current FIR lattice implementation
    </div>

    <b>Active stages:</b> {active_stages}

    &nbsp;&nbsp;&nbsp;

    <b>Active reflection coefficients:</b> {K_text}

    <br><br>

    Equivalent direct-form FIR coefficients:

    <div class="lat-equation">
    h = [{h_text}]
    </div>

    Maximum
    |y<sub>direct</sub>[n] − y<sub>lattice</sub>[n]|:

    <b>{maximum_difference:.3e}</b>

    <table style="margin-top:7px;font-size:12.5px;border-collapse:collapse;">
    {stage_rows}
    </table>

    </div>

    </div>
    """

    # ========================================================
    # SYNCHRONOUS REDRAW
    # ========================================================

    fig1.canvas.draw()
    fig2.canvas.draw()
    fig3.canvas.draw()
    fig4.canvas.draw()

# ============================================================
# OBSERVERS
# ============================================================

stage_slider.observe(update,names='value')

K1_slider.observe(update,names='value')
K2_slider.observe(update,names='value')
K3_slider.observe(update,names='value')
K4_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(fig1.canvas)

display(controls)

display(fig2.canvas)

display(fig3.canvas)

display(fig4.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()